In [1]:
from typing import Tuple

import numpy as np


import time

import numpy as np

import  rigid
import tifffile
import matplotlib.pyplot as plt
import numpy as np

In [3]:
#  Load the movie

mcorr_tif_path='D:/Analysis_2P/Data/Analysis/Scnn1aAi14_A2M0/01192024/run2/mesmerize/c6b67a57-09ed-497a-8d61-4e4752cc6ade/mcorr_u16.tiff'
mcorr_tif_file = tifffile.TiffFile(mcorr_tif_path)
mcorr_tif = mcorr_tif_file.asarray()
mcorr_tif_file.close()

zstack_tiff_file = tifffile.TiffFile("D:/Analysis_2P/Data/Scnn1aAi14_A2M0/01192024/ZSeries-01192024-1323-002/ZSeries-01192024-1323-002_Cycle00001_Ch1_000001.ome.tif")
zstack_tiff = zstack_tiff_file.asarray()
zstack_tiff_file.close()
mcorr_tif.dtype, zstack_tiff.dtype

(dtype('float32'), dtype('uint16'))

In [4]:
def compute_zpos(Zreg, ops, Treg):
    """Compute z position of frames given z-stack Zreg and Treg array.

    Parameters
    ----------
    Zreg : 3D array
        size [nplanes x Ly x Lx], z-stack
    ops : dictionary
        Parameters including "smooth_sigma", "Ly", "Lx", "batch_size", etc.
    Treg : 3D array
        size [nFrames x Ly x Lx], time-series stack to be registered

    Returns
    -------
    ops_orig
    zcorr
    """
    # nbatch = ops["batch_size"]
    Ly = ops["Ly"]
    Lx = ops["Lx"]

    ops_orig = ops.copy()
    ops["nonrigid"] = False
    nplanes, zLy, zLx = Zreg.shape
    if Zreg.shape[1] > Ly or Zreg.shape[2] != Lx:
        Zreg = Zreg[:, :Ly, :Lx]

    nFrames, _, _ = Treg.shape
    zcorr = np.zeros((Zreg.shape[0], nFrames), np.float32)
    t0 = time.time()

    refAndMasks = []
    for Z in Zreg:
        # if ops["1Preg"]:
        #     Z = Z.astype(np.float32)
        #     Z = Z[np.newaxis, :, :]
        #     if ops["pre_smooth"]:
        #         Z = utils.spatial_smooth(Z, int(ops["pre_smooth"]))
        #     Z = utils.spatial_high_pass(Z, int(ops["spatial_hp_reg"]))
        #     Z = Z.squeeze()

        maskMul, maskOffset = rigid.compute_masks(
            refImg=Z,
            maskSlope= 3 * ops["smooth_sigma"],
        )
        cfRefImag = rigid.phasecorr_reference(refImg=Z, smooth_sigma=ops["smooth_sigma"])
        cfRefImag = cfRefImag[np.newaxis, :, :]
        refAndMasks.append((maskMul, maskOffset, cfRefImag))

    for nfr in range(nFrames):
        data = Treg[nfr:nfr + 1]  # Get the current frame
        inds = np.array([nfr])     # Index of the current frame

        for z, ref in enumerate(refAndMasks):
            # if ops["1Preg"]:
            #     data = data.astype(np.float32)
            #     if ops["pre_smooth"]:
            #         data = utils.spatial_smooth(data, int(ops["pre_smooth"]))
            #     data = utils.spatial_high_pass(data, int(ops["spatial_hp_reg"]))

            maskMul, maskOffset, cfRefImg = ref
            cfRefImg = cfRefImg.squeeze()

            _, _, zcorr[z, inds] = rigid.phasecorr(
                data=rigid.apply_masks(data=data, maskMul=maskMul, maskOffset=maskOffset),
                cfRefImg=cfRefImg,
                maxregshift=ops["maxregshift"],
                smooth_sigma_time=ops["smooth_sigma_time"],
            )

        if nfr % 10 == 0:
            print(f"{z + 1} planes, {nfr + 1}/{nFrames} frames, {time.time() - t0:.2f} sec.")

    print(f"{z + 1} planes, {nFrames}/{nFrames} frames, {time.time() - t0:.2f} sec.")
    ops_orig["zcorr"] = zcorr
    return ops_orig, zcorr


In [9]:
ops = {
    'batch_size': 500,           # Number of frames to process at a time
    'Ly': 1024,                   # Height of the frames in Treg
    'Lx': 1024,                   # Width of the frames in Treg
    'smooth_sigma': 1.0,         # Standard deviation for Gaussian smoothing of Z-stack
    '1Preg': False,              # False for two-photon data
    'pre_smooth': 0,             # Pre-smoothing is not typically used for 2P data
    'spatial_hp_reg': 0,         # High-pass filtering is not typically used for 2P data
    'spatial_taper': 50,         # Width of the taper applied to the edges of the frames
    'maxregshift': 20,           # Maximum allowed shift in pixels during registration
    'smooth_sigma_time': 0,      # Standard deviation for temporal smoothing of registration shifts
    'nonrigid': False,           # False for rigid registration
    'nframes': 1200,            # Total number of frames in Treg (adjust based on your dataset)
}
Zreg=zstack_tiff.astype(np.float32)
Treg = mcorr_tif[:30].astype(np.float32)

In [10]:

ops_orig, zcorr = compute_zpos(Zreg,ops,Treg) 

41 planes, 1/30 frames, 8.75 sec.
41 planes, 11/30 frames, 22.39 sec.
41 planes, 21/30 frames, 36.13 sec.
41 planes, 30/30 frames, 48.34 sec.


In [7]:
zcorr.shape

(41, 30)

In [11]:
z_positions = np.argmax(zcorr, axis=0)
print(z_positions)

[ 5 10 17 10 31 37 37 14  2  4 29 28 30 35  4 35  8 23 31  0 16  7 39 29
  7 35  9 24 10 27]
